In [1]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui.widgets import PushButton
from qtpy.QtCore import QTimer

In [ ]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = tif.series[0]
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})
        page = tif.pages[0]

        def _rational(name):
            tag = page.tags.get(name)
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = _rational("XResolution")
        yres = _rational("YResolution")
        resunit_tag = page.tags.get("ResolutionUnit")
        resunit = int(resunit_tag.value) if resunit_tag is not None else None

    meta = {
        "axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit,
    }
    return arr, meta



In [ ]:

def save_image(path, arr, meta):
    """Write `arr` to `path`, preserving the original pixel size / spacing /
    unit metadata.

    The channel / slice / frame counts are recomputed from `arr` (rather than
    copied from the source metadata) so the ImageJ header stays correct even
    when the channel count changes, e.g. when we add a 4th `valid` channel to a
    previously 3-channel stack.
    """
    ij = meta["imagej"]
    axes = meta["axes"]
    md = {"axes": axes}

    # Carry through acquisition metadata that does not depend on array shape.
    for key in ("spacing", "unit", "finterval", "fps", "mode"):
        if key in ij:
            md[key] = ij[key]

    # Recompute shape-dependent counts from the array actually being written.
    for ax, name in (("Z", "slices"), ("C", "channels"), ("T", "frames")):
        if ax in axes:
            md[name] = arr.shape[axes.index(ax)]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    tifffile.imwrite(path, arr, **kwargs)


In [ ]:
# Curate masks one at a time. A single napari viewer is open at any moment,
# showing the full 3D stack (use the z-slider to move through slices). Edit the
# `mask` layer, then click "Save & Next" (or close the window): the curated mask
# is written back into the mask channel and the SAME file is overwritten in
# place (pixel-size metadata preserved). The viewer then reopens with the next
# image automatically.
#
# In a notebook the Qt event loop is already running, so we can't use a blocking
# `for` loop (that opens every viewer at once). Instead we chain images together
# through the viewer's close event.

class MaskCurator:
    def __init__(self, images, start_index=0,
                 brightfield_channel=0, fluorescence_channel=1, mask_channel=2):
        self.images = images
        self.index = start_index
        self.brightfield_channel = brightfield_channel
        self.fluorescence_channel = fluorescence_channel
        self.mask_channel = mask_channel

        self.viewer = None
        self.arr = None
        self.meta = None
        self.mask_layer = None
        self._orig_close = None
        self._advancing = False

    def start(self):
        self._open_current()

    def _open_current(self):
        if self.index >= len(self.images):
            print("All images curated.")
            return

        path = self.images[self.index]
        print(f"[{self.index + 1}/{len(self.images)}] Curating {path.name}")

        self.arr, self.meta = read_image_and_meta(path)

        brightfield = self.arr[:, self.brightfield_channel, :, :]
        fluorescence = self.arr[:, self.fluorescence_channel, :, :]
        mask = self.arr[:, self.mask_channel, :, :]

        self.viewer = napari.Viewer(
            title=f"[{self.index + 1}/{len(self.images)}] {path.name}"
        )
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive")
        self.viewer.add_image(fluorescence, name="fluorescence",
                              colormap="green", blending="additive")
        self.mask_layer = self.viewer.add_labels(mask.astype(np.int32),
                                                 name="mask")
        self.viewer.layers.selection.active = self.mask_layer

        next_btn = PushButton(text="Save & Next")
        next_btn.clicked.connect(self.viewer.close)
        self.viewer.window.add_dock_widget(next_btn, area="right",
                                           name="curation")

        # Route the window close (button or X) through our save+advance logic.
        qt_window = self.viewer.window._qt_window
        self._orig_close = qt_window.closeEvent
        qt_window.closeEvent = self._on_close

    def _on_close(self, event):
        if not self._advancing:
            self._advancing = True
            path = self.images[self.index]

            # Write the curated mask back into the mask channel and overwrite
            # the same file; all other channels and pixel metadata are unchanged.
            self.arr[:, self.mask_channel, :, :] = self.mask_layer.data.astype(self.arr.dtype)
            save_image(path, self.arr, self.meta)
            print(f"    saved -> {path}")

            self.index += 1
            # Open the next image once this window has finished closing.
            QTimer.singleShot(200, self._open_next)
        self._orig_close(event)

    def _open_next(self):
        self._advancing = False
        self._open_current()

In [ ]:
masks_folder = Path(r"C:\Users\isobe\The University of Manchester Dropbox\Isobel Taylor-Hearn\masks_to_curate")

images = sorted(masks_folder.glob("*.tif"))
print(f"{len(images)} images found")

58 images found


## Curate masks and overwrite in place

Opens each tif in `masks_folder` one at a time as a **full 3D stack** (use the
napari z-slider to move through slices). Edit the **mask** layer, then click
**"Save & Next"** (or close the window): the curated mask is written back into
the mask channel and the **same file is overwritten** (pixel-size metadata
preserved). The viewer then reopens with the next image automatically.

In [ ]:
# Set start_index to resume part-way through the list if needed.
curator = MaskCurator(images, start_index=0)
curator.start()